In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from copy import deepcopy
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
jobs_raw_df = pd.read_csv("../processed_data/data_isolated_agg.csv", index_col=0)
jobs_raw_df = jobs_raw_df[jobs_raw_df["Num Nodes"] == 8]
inhibitors_raw_df = pd.read_csv("../processed_data/data_inhib_isolated_agg.csv", index_col=0)
inhibitors_raw_df = inhibitors_raw_df[inhibitors_raw_df["Num Nodes"] == 8]
job_inh_raw_df = pd.read_csv("../processed_data/data_inhib_coscheduled_agg.csv", index_col=0)
job_inh_raw_df = job_inh_raw_df[job_inh_raw_df["Num Nodes"] == 8]
pair_raw_df = pd.read_csv("../processed_data/data_coscheduled_agg.csv", index_col=0)
pair_raw_df = pair_raw_df[pair_raw_df["Num Nodes"] == 8]

In [3]:
# Preprocess data #1

# Isolated jobs
jobs_df = pd.DataFrame()
jobs_df["job_id"] = jobs_raw_df["App"]
jobs_df["mpi_time"] = jobs_raw_df["MPI Time Average Mean"]
jobs_df["comm_frac"] = jobs_raw_df["MPI Time Average Mean"] / jobs_raw_df["App Time Average Mean"]
jobs_df["total_msgs"] = jobs_raw_df["Total Messages Sent Mean"]
jobs_df["total_bytes"] = jobs_raw_df["Total Bytes Sent Mean"]

# Isolated inhibitors
inhibitors_df = pd.DataFrame()
columns_to_use = ["Num Nodes", "Inhib Message Size", "Inhib Wait Time (us)", "Inhib Comm Sparsity"]
selected_data = inhibitors_raw_df[columns_to_use]
inhibitors_df["inhib_id"] = (selected_data
                             .astype(str)
                             .agg('_'.join, axis=1))
inhibitors_df["msg_size"] = inhibitors_raw_df["Inhib Message Size"]
inhibitors_df["wait_time"] = inhibitors_raw_df["Inhib Wait Time (us)"]
inhibitors_df["comm_sparsity"] = inhibitors_raw_df["Inhib Comm Sparsity"]
inhibitors_df["mpi_time"] = inhibitors_raw_df["Inhib MPI Time Average Mean"]
inhibitors_df["comm_frac"] = inhibitors_raw_df["Inhib MPI Time Average Mean"] / inhibitors_raw_df["Inhib App Time Average Mean"]
inhibitors_df["total_msgs"] = inhibitors_raw_df["Inhib Total Messages Sent Mean"]
inhibitors_df["total_bytes"] = inhibitors_raw_df["Inhib Total Bytes Sent Mean"]

# Co-scheduled job and inhibitors
job_inh_df = pd.DataFrame()
job_inh_df["job_id"] = job_inh_raw_df["App"]
columns_to_use = ["Num Nodes", "Inhib Message Size", "Inhib Wait Time (us)", "Inhib Comm Sparsity"]
selected_data = job_inh_raw_df[columns_to_use]
job_inh_df["inhib_id"] = (selected_data
                             .astype(str)
                             .agg('_'.join, axis=1))
job_inh_slowdowns = []
for i in job_inh_raw_df.iloc:
    iso_time = (jobs_df[jobs_df["job_id"] == i["App"]])["mpi_time"].max()
    coscheduled_time = i["MPI Time Average Mean"]
    if (coscheduled_time < iso_time):
        slowdown = 1
    else:
        slowdown = coscheduled_time/iso_time
    job_inh_slowdowns.append(slowdown)
job_inh_df["slowdown"] = job_inh_slowdowns

# Co-scheduled jobs
pair_df = pd.DataFrame()
pair_df["jobA_id"] = pair_raw_df["App A"]
pair_df["jobB_id"] = pair_raw_df["App B"]
pair_slowdowns = []
for i in pair_raw_df.iloc:
    iso_time = (jobs_df[jobs_df["job_id"] == i["App A"]])["mpi_time"].max()
    coscheduled_time = i["App A MPI Time Average Mean"]
    if (coscheduled_time < iso_time):
        slowdown = 1
    else:
        slowdown = coscheduled_time/iso_time
    pair_slowdowns.append(slowdown)
pair_df["slowdown_A"] = pair_slowdowns

In [4]:
# Feature columns
job_feat_cols = ["mpi_time", "comm_frac", "total_msgs", "total_bytes"]
inh_config_cols = ["msg_size", "wait_time", "comm_sparsity"]
inh_prof_cols = ["mpi_time", "comm_frac", "total_msgs", "total_bytes"]

# Combine inhibitor config + profiling into one 7‑dim vector
inhibitors_df["inh_features"] = inhibitors_df.apply(
    lambda row: np.concatenate([row[inh_config_cols].values, row[inh_prof_cols].values]),
    axis=1
)

# -----------------------------
# 1. Build per‑job inhibitor data (skip missing)
# -----------------------------
job_to_inhibitor_data = {}   # job_id -> list of (inhib_id, slowdown, inh_features)
skipped = 0

for job_id in jobs_df["job_id"].unique():
    job_data = job_inh_df[job_inh_df["job_id"] == job_id]
    inhib_list = []
    for _, row in job_data.iterrows():
        inh_id = row["inhib_id"]
        inh_match = inhibitors_df[inhibitors_df["inhib_id"] == inh_id]
        if len(inh_match) == 0 or pd.isna(row["slowdown"]):
            skipped += 1
            continue
        inhib_list.append({
            "inhib_id": inh_id,
            "slowdown": row["slowdown"],
            "inh_feat": inh_match["inh_features"].values[0]
        })
    if len(inhib_list) > 0:
        job_to_inhibitor_data[job_id] = inhib_list
    else:
        print(f"Warning: Job {job_id} has no valid inhibitor data")

print(f"Skipped {skipped} missing experiments")
valid_jobs = list(job_to_inhibitor_data.keys())
print(f"Valid jobs: {len(valid_jobs)}")

# -----------------------------
# 2. Normalise features
# -----------------------------
# Job isolated features
job_feat_matrix = jobs_df[jobs_df["job_id"].isin(valid_jobs)][job_feat_cols].values
job_scaler = StandardScaler().fit(job_feat_matrix)

# Inhibitor features (7‑dim)
all_inh_feats = np.vstack([np.array(d["inh_feat"]) for data in job_to_inhibitor_data.values() for d in data])
inh_scaler = StandardScaler().fit(all_inh_feats)

# Target slowdown (from pair experiments)
pair_df_filtered = pair_df[
    pair_df["jobA_id"].isin(valid_jobs) & 
    pair_df["jobB_id"].isin(valid_jobs)
]
slowdown_scaler = StandardScaler().fit(pair_df_filtered[["slowdown_A"]].values)

# -----------------------------
# 3. Build augmented dataset: each row = (jobA, inhA, jobB, inhB, pair_slowdown)
# -----------------------------
# For each pair, generate all possible inhibitor combinations (or a large random subset)
# We'll store: jobA_feat, inhA_feat, jobB_feat, inhB_feat, target

def build_augmented_dataset(pair_df, job_to_inhibitor_data, max_samples_per_pair=500):
    """
    Returns list of samples as dicts.
    max_samples_per_pair limits total combos per pair to avoid memory blow‑up.
    """
    samples = []
    for _, row in pair_df.iterrows():
        jobA = row["jobA_id"]
        jobB = row["jobB_id"]
        target = row["slowdown_A"]
        
        if jobA not in job_to_inhibitor_data or jobB not in job_to_inhibitor_data:
            continue
        
        inhA_list = job_to_inhibitor_data[jobA]
        inhB_list = job_to_inhibitor_data[jobB]
        
        # If too many combos, sample randomly
        nA, nB = len(inhA_list), len(inhB_list)
        total_combos = nA * nB
        if total_combos > max_samples_per_pair:
            # Randomly sample indices
            idx_pairs = np.random.choice(total_combos, max_samples_per_pair, replace=False)
            for idx in idx_pairs:
                iA = idx // nB
                iB = idx % nB
                samples.append({
                    'jobA_feat': jobs_df[jobs_df["job_id"] == jobA][job_feat_cols].values[0],
                    'inhA_feat': inhA_list[iA]["inh_feat"],
                    'jobB_feat': jobs_df[jobs_df["job_id"] == jobB][job_feat_cols].values[0],
                    'inhB_feat': inhB_list[iB]["inh_feat"],
                    'target': target
                })
        else:
            for inhA in inhA_list:
                for inhB in inhB_list:
                    samples.append({
                        'jobA_feat': jobs_df[jobs_df["job_id"] == jobA][job_feat_cols].values[0],
                        'inhA_feat': inhA["inh_feat"],
                        'jobB_feat': jobs_df[jobs_df["job_id"] == jobB][job_feat_cols].values[0],
                        'inhB_feat': inhB["inh_feat"],
                        'target': target
                    })
    return samples

# Build training data using only a subset of job IDs (for later generalisation test)
train_job_ids = valid_jobs[:7]   # e.g., first 7 apps for training
test_job_ids = [j for j in valid_jobs if j not in train_job_ids]

train_pair_df = pair_df_filtered[
    pair_df_filtered["jobA_id"].isin(train_job_ids) & 
    pair_df_filtered["jobB_id"].isin(train_job_ids)
]
test_pair_df = pair_df_filtered[
    ~(pair_df_filtered["jobA_id"].isin(train_job_ids) & 
      pair_df_filtered["jobB_id"].isin(train_job_ids))
]

train_samples = build_augmented_dataset(train_pair_df, job_to_inhibitor_data, max_samples_per_pair=500)
test_samples = build_augmented_dataset(test_pair_df, job_to_inhibitor_data, max_samples_per_pair=500)

print(f"Training samples: {len(train_samples)}, Test samples: {len(test_samples)}")

Skipped 12 missing experiments
Valid jobs: 10
Training samples: 14000, Test samples: 13500


In [5]:
# -----------------------------
# 4. PyTorch Dataset
# -----------------------------
class PairAugmentedDataset(Dataset):
    def __init__(self, samples, job_scaler, inh_scaler, slowdown_scaler):
        self.samples = samples
        self.job_scaler = job_scaler
        self.inh_scaler = inh_scaler
        self.slowdown_scaler = slowdown_scaler
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        jobA = self.job_scaler.transform(s['jobA_feat'].reshape(1, -1)).astype(np.float32).flatten()
        inhA = self.inh_scaler.transform(s['inhA_feat'].reshape(1, -1)).astype(np.float32).flatten()
        jobB = self.job_scaler.transform(s['jobB_feat'].reshape(1, -1)).astype(np.float32).flatten()
        inhB = self.inh_scaler.transform(s['inhB_feat'].reshape(1, -1)).astype(np.float32).flatten()
        target = self.slowdown_scaler.transform([[s['target']]]).astype(np.float32).flatten()[0]
        
        return {
            'xA': torch.tensor(np.concatenate([jobA, inhA])),  # 4+7 = 11
            'xB': torch.tensor(np.concatenate([jobB, inhB])),
            'jobA_feat': torch.tensor(jobA),
            'jobB_feat': torch.tensor(jobB),
            'target': torch.tensor(target)
        }

# Create datasets
train_dataset = PairAugmentedDataset(train_samples, job_scaler, inh_scaler, slowdown_scaler)
test_dataset = PairAugmentedDataset(test_samples, job_scaler, inh_scaler, slowdown_scaler)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# -----------------------------
# 5. Model definition
# -----------------------------
class JobInhibitorEncoder(nn.Module):
    """Maps (job_feat, inh_feat) -> latent vector"""
    def __init__(self, input_dim=11, latent_dim=16, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, latent_dim),
            nn.BatchNorm1d(latent_dim),
        )
        
    def forward(self, x):
        return self.net(x)

class PairPredictor(nn.Module):
    """Combines two latent vectors and isolated features to predict slowdown"""
    def __init__(self, latent_dim=16, job_feat_dim=4, hidden_dim=16, dropout=0.3):
        super().__init__()
        input_dim = latent_dim * 2 + job_feat_dim * 2
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, zA, zB, jobA_feat, jobB_feat):
        x = torch.cat([zA, zB, jobA_feat, jobB_feat], dim=-1)
        return self.net(x)

class FullAugmentedModel(nn.Module):
    def __init__(self, input_dim=11, latent_dim=16, job_feat_dim=4, dropout=0.3):
        super().__init__()
        self.encoder = JobInhibitorEncoder(input_dim, latent_dim, dropout)
        self.predictor = PairPredictor(latent_dim, job_feat_dim, dropout=dropout)
        
    def forward(self, xA, xB, jobA_feat, jobB_feat):
        zA = self.encoder(xA)
        zB = self.encoder(xB)
        return self.predictor(zA, zB, jobA_feat, jobB_feat)

In [7]:
# -----------------------------
# 6. Training
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print(f"Device: {device}")

model = FullAugmentedModel(input_dim=11, latent_dim=16, job_feat_dim=4, dropout=0.3).to(device)

# Weight initialization (Xavier with reduced gain)
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight, gain=0.5)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
model.apply(init_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=15, factor=0.5, min_lr=1e-6)
loss_fn = nn.MSELoss()

epochs = 200
best_val_loss = float('inf')
best_state = None

# Split training data into train/val for monitoring
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [train_size, val_size])
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        pred = model(batch['xA'], batch['xB'], batch['jobA_feat'], batch['jobB_feat'])
        loss = loss_fn(pred.squeeze(), batch['target'])
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            pred = model(batch['xA'], batch['xB'], batch['jobA_feat'], batch['jobB_feat'])
            loss = loss_fn(pred.squeeze(), batch['target'])
            val_loss += loss.item()
    val_loss /= len(val_loader)
    
    scheduler.step(val_loss)
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = deepcopy(model.state_dict())

model.load_state_dict(best_state)

Device: cuda
Epoch 0: Train Loss=0.9827, Val Loss=0.4952
Epoch 20: Train Loss=0.1581, Val Loss=0.1221
Epoch 40: Train Loss=0.0952, Val Loss=0.0537
Epoch 60: Train Loss=0.0839, Val Loss=0.0356
Epoch 80: Train Loss=0.0734, Val Loss=0.0293
Epoch 100: Train Loss=0.0700, Val Loss=0.0274
Epoch 120: Train Loss=0.0689, Val Loss=0.0264
Epoch 140: Train Loss=0.0667, Val Loss=0.0262
Epoch 160: Train Loss=0.0625, Val Loss=0.0226
Epoch 180: Train Loss=0.0615, Val Loss=0.0224


<All keys matched successfully>

In [8]:
# -----------------------------
# 7. Evaluation
# -----------------------------
def evaluate_model(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            pred = model(batch['xA'], batch['xB'], batch['jobA_feat'], batch['jobB_feat'])
            all_preds.append(pred.cpu().numpy())
            all_targets.append(batch['target'].cpu().numpy())
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    
    # Inverse scale
    preds_orig = slowdown_scaler.inverse_transform(all_preds.reshape(-1,1))
    targets_orig = slowdown_scaler.inverse_transform(all_targets.reshape(-1,1))
    return preds_orig, targets_orig

train_preds, train_targets = evaluate_model(model, train_loader)
test_preds, test_targets = evaluate_model(model, test_loader)

def metrics(preds, targets):
    mae = np.mean(np.abs(preds - targets))
    mape = np.mean(np.abs((preds - targets) / (targets + 1e-8))) * 100
    ss_res = np.sum((preds - targets)**2)
    ss_tot = np.sum((targets - np.mean(targets))**2)
    r2 = 1 - ss_res/(ss_tot + 1e-8)
    return mae, mape, r2

train_mae, train_mape, train_r2 = metrics(train_preds, train_targets)
test_mae, test_mape, test_r2 = metrics(test_preds, test_targets)

print("\n=== Results ===")
print(f"Train MAE: {train_mae:.4f}, MAPE: {train_mape:.2f}%, R²: {train_r2:.4f}")
print(f"Test  MAE: {test_mae:.4f}, MAPE: {test_mape:.2f}%, R²: {test_r2:.4f}")


=== Results ===
Train MAE: 0.0179, MAPE: 1.64%, R²: 0.9682
Test  MAE: 0.1495, MAPE: 12.44%, R²: -0.1805
